# Exercise 1: Retention-basin information extraction

Unstructured reports and papers often mention flood-retention basins, polders, and reservoirs without a structured database provided. This exercise uses **DLLMForge** package and **Azure OpenAI model** to extract retention basin information from PDF, and output a structured JSON that matches a user-defined schema, which can further be used for your own modeling or analysis works.

You will:

1. Define and inspect a hand-written Pydantic schema.
2. Convert a PDF with Docling (sections, markdown tables, `[PAGE N]` tags).
3. Extract records with `DoclingInfoExtractor`.
4. Compare the JSON to 'ground truth' values in `README.md` for Eupen and La Gileppe.

Prerequisites: course `uv sync` / hydrology-env verify is green, and Azure OpenAI variables are in the repo-root `.env`. Never paste API keys into this notebook.

## Step 1: Imports and initialize LLMs


In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from dllmforge.IE_agent_extractor_docling import DoclingDocumentProcessor, DoclingInfoExtractor
from dllmforge.langchain_api import LangchainAPI

EXERCISE_DIR = Path.cwd()
if not (EXERCISE_DIR / "schema.py").exists():
    EXERCISE_DIR = Path.cwd() / "exercises" / "01_information_extraction"
if str(EXERCISE_DIR) not in sys.path:
    sys.path.insert(0, str(EXERCISE_DIR))

from schema import WaterRetentionBasinDocument

print(f"Exercise dir: {EXERCISE_DIR}")
print("Imports OK")

In [ ]:
load_dotenv()
if not os.getenv("AZURE_OPENAI_ENDPOINT") and os.getenv("AZURE_OPENAI_API_BASE"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = os.environ["AZURE_OPENAI_API_BASE"]

llm_api = LangchainAPI(model_provider="azure-openai", temperature=0.1)
print("Azure OpenAI ready (", os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"), ")")

## Step 2: Define extraction schema

The second step is to know what information you want to extract, and define a pydantic schema. The schema is the contract: if a field is missing, the model will not return it. Let's print out the schema to inspect what information we are going to extract.

In [ ]:
print("\nWaterRetentionBasin fields:")
from schema import WaterRetentionBasin

for name, info in WaterRetentionBasin.model_fields.items():
    print(f"  - {name}: {info.annotation}")

## Step 3: Docling preprocess (sections, tables, pages)

`DoclingDocumentProcessor` converts the PDF to tagged markdown: `[PAGE N]`, `[SECTION: …]`, `[TABLE: …]`, `[FIGURE: …]`. Tables stay as markdown grids (not PyMuPDF cell soup). `[PAGE N]` is the PDF sheet index, not the journal page number. `source_page` is filled later from `source_quote`, not by the LLM.

Note: Docling does not only parse PDF text. It runs neural layout models so it can tell paragraphs from tables, figures, and reading order. Those model weights are not in the pip/uv package. On the first `process_document()` call, Docling fetches them from Hugging Face into a local cache. After that, later runs reuse the cache. No Hugging Face download unless the cache is cleared or the model version changes.

In [ ]:
pdf_path = EXERCISE_DIR / "data" / "pdfs" / "nhess-15-365-2015.pdf"
output_dir = EXERCISE_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)

doc_processor = DoclingDocumentProcessor(
    input_dir=str(EXERCISE_DIR / "data" / "pdfs"),
    file_pattern="*.pdf",
    output_dir=str(output_dir),
)
processed = doc_processor.process_document(pdf_path)
print(f"File: {pdf_path.name}")
print(
    f"Pages: {processed.metadata.get('page_count')}, "
    f"characters: {processed.metadata.get('text_length')}, "
    f"tables: {processed.metadata.get('num_tables')}"
)
print("--- first 800 characters ---")
print(processed.content[:800])
table_at = processed.content.find("[TABLE:")
if table_at >= 0:
    print("--- Docling table preview ---")
    print(processed.content[table_at:table_at + 600])

## Step 4: Build the extractor

**1) Define the system prompt** for the extraction task. Prefer markdown **table** values over rounded prose; copy one table row or one sentence as `source_quote`; do not set `source_page`.

**2) Reuse the Docling processor** from Step 3 (`doc_processor`).

**3) Create `DoclingInfoExtractor`.** It combines the schema, the LLM, the Docling processor, and the system prompt. After the LLM returns, DLLMForge locates `source_quote` in the element index and fills `source_page`.

In [ ]:
system_prompt = """Extract every water retention basin, polder, or multi-purpose reservoir
mentioned in the document into the provided schema.

Rules:
- One list entry per distinct site. Do not merge two dams or basins (e.g. keep Eupen and La Gileppe separate).
- Only extract values that are explicitly stated. Use null if a field is missing or uncertain.
- Convert all volumes to cubic metres (m³). 1 hm³ = 1e6 m³, 1 Mm³ = 1e6 m³.
- river_name is the local river; river_basin is the main basin (e.g. Meuse).
- Prefer markdown table values over rounded prose when they conflict.
- Expand table scientific notation to a number (25 × 10^6 m³ → 25000000).
- Copy source_quote from one table row or one sentence (not several blocks concatenated). Do not paraphrase.
- Do not set source_page; it is filled after extraction from the quote.

Return JSON only, matching the schema keys.
"""

extractor = DoclingInfoExtractor(
    output_schema=WaterRetentionBasinDocument,
    llm_api=llm_api,
    system_prompt=system_prompt,
    chunk_size=80000,
    chunk_overlap=10000,
    doc_processor=doc_processor,
)
print("Extractor ready")

## Step 5: Extract from one PDF

This cell calls Azure GPT and may take a minute.

In [ ]:
results = extractor.process_document(processed)
output_path = output_dir / f"{pdf_path.stem}_extracted.json"
extractor.save_results(results, output_path)

print(f"Saved {len(results)} result object(s) to {output_path}")
print(json.dumps(results, indent=2, ensure_ascii=False, default=str))

## Step 6: Inspect metadata vs evidence, then check 'groundtruth'values

- `metadata` comes from the Docling processor (file name, page count, table/figure counts). The locator index is used internally and is not written to the JSON.
- `source_quote` comes from the LLM. `source_page` / `quote_verified` are filled by DLLMForge from that quote. Open the PDF sheet and confirm the quote is there.

Groundtruth values below are **approximate** (Bruwier et al., 2015). Volumes may be wrong if the model forgot to convert hm³ / Mm³ to m³.

In [ ]:
basins = []
for result in results:
    print("Processor metadata:", result.get("metadata"))
    basins.extend(result.get("basins") or [])

print(f"\nExtracted {len(basins)} basin(s)")
for basin in basins:
    print("-" * 60)
    print(f"name:         {basin.get('name')}")
    print(f"country:      {basin.get('country')}")
    print(f"river_name:   {basin.get('river_name')}")
    print(f"river_basin:  {basin.get('river_basin')}")
    print(f"control:      {basin.get('basin_control_type')}")
    print(f"total m3:     {basin.get('storage_volume_total_m3')}")
    print(f"flood m3:     {basin.get('flood_water_storage_volume_m3')}")
    print(f"source_page:  {basin.get('source_page')}")
    print(f"quote_ok:     {basin.get('quote_verified')}")
    print(f"source_quote: {basin.get('source_quote')}")

GOLD_VALUES = [
    {"name": "Eupen dam", "country": "Belgium", "river_name": "Vesdre", "river_basin": "Meuse", "basin_control_type": "controlled"},
    {"name": "La Gileppe dam", "country": "Belgium", "river_name": "Vesdre", "river_basin": "Meuse", "basin_control_type": "controlled"},
]
print("\nGold-value check:")
print(f"{'expected':<22} {'extracted name':<28} {'country':<12} {'river':<12} {'type':<12}")
for gold in GOLD_VALUES:
    needle = gold["name"].split()[0].lower()
    match = next((b for b in basins if needle in (b.get("name") or "").lower()), {})
    print(
        f"{gold['name']:<22} "
        f"{str(match.get('name') or 'NOT FOUND'):<28} "
        f"{str(match.get('country') or ''):<12} "
        f"{str(match.get('river_name') or ''):<12} "
        f"{str(match.get('basin_control_type') or ''):<12}"
    )